# Lab 9 – Modern NLP Final Project Development
**Học phần:** Xử lý Ngôn ngữ Tự nhiên (1230554) · **Dự án:** SSA-VN – Trợ lý hỗ trợ sinh viên bằng tiếng Việt

**Hình thức:** nhóm 3 sinh viên · **Thẻ và dữ liệu chung: MSSV nhóm trưởng `24DH190314`**

| Vai trò | Họ tên | MSSV |
|---|---|---|
| Nhóm trưởng | Nguyễn Hoàng Phúc | 24DH190314 |
| Thành viên | Nguyễn Bảo Duy | 24DH190309 |
| Thành viên | Lê Trần Nhật Huy | 24DH190312 |

**Pipeline:** phân loại chủ đề → truy hồi (BM25 / dense / hybrid) → RAG có từ chối → đánh giá, ablation, phân tích lỗi.

**Cách chạy:** đặt notebook cùng thư mục với `student_pack_24DH190314/` và `lab9_tools.py`, rồi *Restart & Run All*.
Phần bắt buộc chạy được trên CPU. Riêng C4 (LoRA, nâng cao) chỉ chạy khi có GPU (Colab T4).

| Phần | Nội dung | Yêu cầu của đề |
|---|---|---|
| A | Dữ liệu & tiền xử lý | A1, A2, A3 |
| B | Baseline cổ điển (TF-IDF + LR, BM25) | B1, B2, B3 |
| C | Embedding, dense, hybrid, LoRA | C1, C2, C3, C4 (nâng cao) |
| D | LLM: prompting, RAG, từ chối trả lời | D1, D2, D3 |
| E | Tính tay LoRA / DPO / metric rồi kiểm bằng code | E1, E2, E3 (mỗi thành viên một thẻ) |
| F | Ablation, phân tích lỗi, câu hỏi tự viết, hạn chế | F1, F2, F3, F4 |

In [1]:
# Cài thư viện (chỉ cần chạy 1 lần; trên Colab thì bỏ dấu # ở dòng dưới)
# !pip -q install pandas scikit-learn sentence-transformers transformers peft

In [2]:
import json, math, re, pathlib, unicodedata
from collections import Counter
import numpy as np, pandas as pd
import torch

pd.set_option("display.max_colwidth", 120, "display.width", 200)

# ---- Nhóm: toàn bộ sản phẩm chung dùng thẻ của NHÓM TRƯỞNG ----
MSSV = "24DH190314"
NHOM = [   # (họ tên, MSSV, vai trò, phần phụ trách)  <- SỬA cột cuối cho đúng phân công thực tế của nhóm
    ("Nguyễn Hoàng Phúc", "24DH190314", "Nhóm trưởng", "Phần A, B; ghép notebook, results.json, báo cáo; Phần E theo thẻ riêng"),
    ("Nguyễn Bảo Duy",    "24DH190309", "Thành viên",  "Phần C (dense, hybrid, phân loại embedding) và F1 (ablation); Phần E theo thẻ riêng"),
    ("Lê Trần Nhật Huy",   "24DH190312", "Thành viên",  "Phần D (prompting, RAG, từ chối) và F2–F4; Phần E theo thẻ riêng"),
]

PACK = pathlib.Path(f"student_pack_{MSSV}")          # tạo bằng: python lab9_tools.py pack --mssv 24DH190314
card = json.load(open(PACK / "personal_card.json", encoding="utf-8"))

SEED = card["seed"]                                  # seed của thẻ dùng cho MỌI bước ngẫu nhiên
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("MSSV nhóm trưởng:", card["mssv"], "| check_code:", card["check_code"], "| seed:", SEED, "| device:", DEVICE)
print("noise:", card["noise"], "| topic_pair:", card["topic_pair"], "| ablation:", card["ablation"])
print("embed_model:", card["embed_model"], "| clf_backbone:", card["clf_backbone"])
print("lora:", card["lora"], "| fewshot_k:", card["fewshot_k"], "| rag_top_k:", card["rag_top_k"])

MSSV nhóm trưởng: 24DH190314 | check_code: 927D30B4 | seed: 58202 | device: cpu
noise: code_mix | topic_pair: ['thi_diem', 'thu_vien_cntt'] | ablation: A4
embed_model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 | clf_backbone: xlm-roberta-base
lora: {'r': 8, 'alpha': 16, 'dropout': 0.1, 'target_modules': ['query', 'value']} | fewshot_k: 6 | rag_top_k: 2


## Phần A – Dữ liệu & tiền xử lý
### A1. Nạp dữ liệu, phân bố nhãn, độ dài câu, 5 câu clean cạnh 5 câu noisy

In [3]:
def doc(ten_file):
    return pd.read_csv(PACK / ten_file)

kb = doc("kb_passages.csv")                                   # 80 đoạn quy định
tr, te, te_noisy = doc("clf_train.csv"), doc("clf_test.csv"), doc("clf_test_noisy.csv")      # phân loại
dev, rq, rq_noisy = doc("retr_dev.csv"), doc("retr_test.csv"), doc("retr_test_noisy.csv")   # truy hồi
fewshot, llm_eval, ood = doc("fewshot_pool.csv"), doc("llm_eval_30.csv"), doc("ood_10.csv")  # LLM

print("train", tr.shape, "| test", te.shape, "| test_noisy", te_noisy.shape, "| kb", kb.shape)
print("dev", dev.shape, "| retr_test", rq.shape, "| retr_test_noisy", rq_noisy.shape)

TOPICS = sorted(tr.topic.unique())                            # 8 chủ đề
print("\nPhân bố nhãn (số câu hỏi mỗi chủ đề):")
print(pd.DataFrame({"train": tr.topic.value_counts(), "test": te.topic.value_counts()}))

print("\nĐộ dài câu trung bình (số âm tiết = số từ tách theo khoảng trắng):")
for ten, df in [("train", tr), ("test", te), ("test_noisy", te_noisy)]:
    print(f"  {ten:10s}: {df.text.str.split().str.len().mean():.2f} âm tiết/câu")

# 5 câu clean đặt cạnh 5 câu noisy (cùng qid), lấy trong các câu thực sự bị nhiễu làm thay đổi
bi_doi = np.where(te.text.values != te_noisy.text.values)[0]
print(f"\nNhiễu '{card['noise']}' làm thay đổi {len(bi_doi)}/{len(te)} câu test ({len(bi_doi) / len(te):.1%}).")
chon = np.random.RandomState(SEED).choice(bi_doi, 5, replace=False)
print(pd.DataFrame({"qid": te.qid.values[chon], "clean": te.text.values[chon], "noisy": te_noisy.text.values[chon]}).to_string(index=False))

train (1392, 8) | test (528, 8) | test_noisy (528, 8) | kb (80, 5)
dev (116, 8) | retr_test (132, 8) | retr_test_noisy (132, 8)

Phân bố nhãn (số câu hỏi mỗi chủ đề):
                  train  test
topic                        
ctsv                168    72
dang_ky_hoc_phan    168    72
hoc_bong            168    72
hoc_phi             168    72
ky_tuc_xa           168    72
thi_diem            192    48
thu_vien_cntt       192    48
tot_nghiep          168    72

Độ dài câu trung bình (số âm tiết = số từ tách theo khoảng trắng):
  train     : 14.11 âm tiết/câu
  test      : 13.69 âm tiết/câu
  test_noisy: 13.20 âm tiết/câu

Nhiễu 'code_mix' làm thay đổi 239/528 câu test (45.3%).
    qid                                                                          clean                                                                                noisy
Q047-12               Có deadline cho việc nộp khóa luận tốt nghiệp không, là bao giờ?                  Có deadline cho việc submit khóa luậ

**Nhận xét A1.** Tám chủ đề, mỗi chủ đề 10 mục (quy định) trong KB; mỗi mục có khoảng 24 câu hỏi. Số câu của `thi_diem` và `thu_vien_cntt` ở test ít hơn (48 so với 72) vì hai chủ đề trong `topic_pair` chỉ có 2 mục test, các chủ đề khác có 3 mục. Câu hỏi ngắn, khoảng 13–14 âm tiết. Nhiễu `code_mix` thay từ khóa tiếng Việt bằng tiếng Anh (*học phí → tuition, đăng ký → register, nộp → submit, hồ sơ → document…*) nên chỉ những câu chứa các từ đó mới bị đổi (khoảng 45 % số câu).

### A2. Kiểm chứng không rò rỉ dữ liệu (tách theo **mục**, không tách theo câu)

In [4]:
# Không mục (item_id) nào được xuất hiện ở cả train và test
assert set(tr.item_id).isdisjoint(set(te.item_id)), "Rò rỉ dữ liệu: item trùng giữa train/test!"
# dev (dùng để chọn α, ngưỡng) lấy từ mục train; retr_test (chỉ để báo cáo) lấy từ mục test
assert set(dev.item_id) <= set(tr.item_id) and set(rq.item_id) <= set(te.item_id)
assert set(dev.item_id).isdisjoint(set(rq.item_id)), "Rò rỉ dev -> test!"
# kho ví dụ few-shot cũng không được chứa mục test
assert set(fewshot.item_id).isdisjoint(set(te.item_id))

print(f"Không rò rỉ: train {tr.item_id.nunique()} mục, test {te.item_id.nunique()} mục, "
      f"mỗi mục có khoảng {len(tr) / tr.item_id.nunique():.0f} câu hỏi.")

Không rò rỉ: train 58 mục, test 22 mục, mỗi mục có khoảng 24 câu hỏi.


**Giải thích A2.** Mỗi *mục* là một quy định. Khoảng 24 câu hỏi của cùng một mục chỉ là các cách diễn đạt khác nhau của cùng một nội dung (nói chuẩn / nói đời thường; hỏi đơn vị / thời hạn / lệ phí). Nếu tách ngẫu nhiên **theo câu**, các câu "anh em" của một câu test đã nằm sẵn trong train, mô hình chỉ cần nhớ mục đó là trả lời đúng → điểm cao giả tạo (rò rỉ dữ liệu). Tách **theo mục** thì mọi câu test đều thuộc quy định mô hình chưa từng thấy, nên điểm số đo đúng khả năng khái quát – giống lúc triển khai thật khi trường ban hành quy định mới.

### A3. Hàm `normalize`: NFC → chữ thường → bỏ dấu câu → gọn khoảng trắng

In [5]:
def normalize(s: str) -> str:
    s = unicodedata.normalize("NFC", str(s)).lower()    # (1) gộp dấu về dạng dựng sẵn NFC, (2) chữ thường
    s = re.sub(r"[^\w\s]", " ", s)                      # (3) bỏ dấu câu
    return re.sub(r"\s+", " ", s).strip()               # (4) gọn khoảng trắng

def tok(s: str):                                        # tách theo âm tiết (khoảng trắng)
    return normalize(s).split()

# Ví dụ thật: một câu hỏi test bắt đầu bằng chữ hoa "Lệ phí" và kết thúc bằng dấu "?"
vi_du = te[te.text.str.startswith("Lệ phí")].iloc[0]
cau_hoi = vi_du.text
doan_dung = kb.text[kb.passage_id == vi_du.gold_passage_id].iloc[0]
print("Câu hỏi       :", cau_hoi)
print("Đoạn quy định :", doan_dung)

tu_tho, tu_chuan = cau_hoi.split(), tok(cau_hoi)
khop_tho = [w for w in tu_tho if w in doan_dung.split()]          # KHÔNG normalize (cả câu hỏi lẫn đoạn)
khop_chuan = [w for w in tu_chuan if w in tok(doan_dung)]         # CÓ normalize
print("\nKhông normalize:", tu_tho, "-> khớp", len(khop_tho), "từ:", khop_tho)
print("Có normalize   :", tu_chuan, "-> khớp", len(khop_chuan), "từ:", khop_chuan)

# NFC: cùng một câu nhưng dấu bị tách rời (dạng NFD, hay gặp khi copy từ PDF / máy Mac)
dang_nfd = unicodedata.normalize("NFD", cau_hoi)
print(f"\nDạng NFD dài {len(dang_nfd)} ký tự, dạng NFC dài {len(cau_hoi)} ký tự; so sánh thẳng bằng nhau? {dang_nfd == cau_hoi}")
print("Sau normalize bằng nhau?", normalize(dang_nfd) == normalize(cau_hoi))

Câu hỏi       : Lệ phí đăng ký học lại là bao nhiêu?
Đoạn quy định : Hướng dẫn đăng ký học lại. Đối tượng áp dụng: sinh viên học phần có điểm dưới mức đạt. Đơn vị tiếp nhận: Phòng Đào tạo. Thời hạn: trước tuần thứ 2 của học kỳ. Lệ phí: bằng học phí theo số tín chỉ học lại.

Không normalize: ['Lệ', 'phí', 'đăng', 'ký', 'học', 'lại', 'là', 'bao', 'nhiêu?'] -> khớp 5 từ: ['Lệ', 'phí', 'đăng', 'ký', 'học']
Có normalize   : ['lệ', 'phí', 'đăng', 'ký', 'học', 'lại', 'là', 'bao', 'nhiêu'] -> khớp 6 từ: ['lệ', 'phí', 'đăng', 'ký', 'học', 'lại']

Dạng NFD dài 45 ký tự, dạng NFC dài 36 ký tự; so sánh thẳng bằng nhau? False
Sau normalize bằng nhau? True


**Giải thích A3.**
1. **NFC**: một chữ tiếng Việt có hai cách mã hóa (ví dụ `ệ` là 1 ký tự, hoặc `e` + dấu mũ + dấu nặng rời). Nhìn giống hệt nhưng máy coi là hai chuỗi khác nhau, nên TF-IDF/BM25 không khớp được. NFC đưa về một dạng duy nhất.
2. **Chữ thường**: `Lệ` và `lệ` thành một từ.
3. **Bỏ dấu câu**: `phí:` / `phí?` / `lại.` không còn dính dấu → khớp với `phí`, `lại`.
4. **Gọn khoảng trắng**: để `split()` không sinh từ rỗng.

Ví dụ trên: không normalize thì từ `lại` của câu hỏi không khớp được vì trong đoạn quy định nó dính dấu chấm (`lại.`), còn `nhiêu?` dính dấu hỏi; sau normalize số từ khớp với đoạn đúng tăng từ 5 lên 6 → BM25 chấm điểm đoạn đúng cao hơn. Cùng câu đó ở dạng NFD (dấu rời) thì so sánh trực tiếp là khác nhau, sau normalize mới bằng nhau.

## Phần B – Baseline cổ điển
### B1. TF-IDF + Logistic Regression (clean và noisy)

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

def cham_phan_loai(y_that, y_doan):
    return {"acc": accuracy_score(y_that, y_doan), "macro_f1": f1_score(y_that, y_doan, average="macro")}

vec = TfidfVectorizer(preprocessor=normalize, ngram_range=(1, 2), sublinear_tf=True)   # từ đơn + cụm 2 từ
Xtr = vec.fit_transform(tr.text)                                                      # chỉ fit trên train
clf = LogisticRegression(max_iter=2000, random_state=SEED).fit(Xtr, tr.topic)

pred_tfidf = {"clean": clf.predict(vec.transform(te.text)), "noisy": clf.predict(vec.transform(te_noisy.text))}
res = {"clean": cham_phan_loai(te.topic, pred_tfidf["clean"]), "noisy": cham_phan_loai(te_noisy.topic, pred_tfidf["noisy"])}
print("TF-IDF + LR:")
print(pd.DataFrame(res).T.round(4))

TF-IDF + LR:
          acc  macro_f1
clean  0.6534    0.6344
noisy  0.5076    0.4871


### B2. BM25 (cài đặt của giảng viên, giữ nguyên) – Recall@1/3/5/10, MRR@10, nDCG@10

In [7]:
class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.tf = [Counter(tok(d)) for d in docs]
        self.dl = np.array([sum(c.values()) for c in self.tf], dtype=float)
        self.avg = self.dl.mean()
        df = Counter(w for c in self.tf for w in c)
        n = len(docs)
        self.idf = {w: math.log(1 + (n - f + 0.5) / (f + 0.5)) for w, f in df.items()}

    def scores(self, q):
        s = np.zeros(len(self.tf))
        for w in tok(q):
            if w not in self.idf:
                continue
            for i, c in enumerate(self.tf):
                f = c.get(w, 0)
                if f:
                    s[i] += self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.dl[i] / self.avg))
        return s

def rank_of_gold(score_vec, gold_idx):
    """Thứ hạng (bắt đầu từ 1) của đáp án đúng; điểm bằng nhau -> tính theo thứ tự chỉ số (tất định)."""
    order = np.lexsort((np.arange(len(score_vec)), -score_vec))
    return int(np.where(order == gold_idx)[0][0]) + 1

def retrieval_metrics(ranks, ks=(1, 3, 5, 10)):
    ranks = np.asarray(ranks)
    out = {f"recall@{k}": float((ranks <= k).mean()) for k in ks}
    out["mrr@10"] = float(np.mean([1 / r if r <= 10 else 0 for r in ranks]))
    out["ndcg@10"] = float(np.mean([1 / math.log2(r + 1) if r <= 10 else 0 for r in ranks]))  # 1 tài liệu đúng -> IDCG = 1
    return out

pid2idx = {p: i for i, p in enumerate(kb.passage_id)}          # passage_id -> số thứ tự dòng trong kb
bm25 = BM25(kb.text.tolist())

# Hai hàm tiện ích dùng chung cho BM25 / dense / hybrid: làm việc trên MA TRẬN điểm [số câu hỏi x 80 đoạn]
def bm25_matrix(texts):
    return np.array([bm25.scores(t) for t in texts])

def eval_matrix(S, df):
    """S[i] = điểm của 80 đoạn với câu hỏi thứ i -> hạng của đoạn đúng -> các metric truy hồi."""
    ranks = [rank_of_gold(S[i], pid2idx[g]) for i, g in enumerate(df.gold_passage_id)]
    return retrieval_metrics(ranks)

B_dev, B_rq, B_rqn = bm25_matrix(dev.text), bm25_matrix(rq.text), bm25_matrix(rq_noisy.text)
ret = {"bm25_clean": eval_matrix(B_rq, rq), "bm25_noisy": eval_matrix(B_rqn, rq_noisy)}
print(pd.DataFrame(ret).T.round(3))

            recall@1  recall@3  recall@5  recall@10  mrr@10  ndcg@10
bm25_clean     0.727     0.841     0.894      0.970   0.804    0.844
bm25_noisy     0.644     0.818     0.894      0.962   0.750    0.801


### B3. Nhiễu `code_mix` làm baseline giảm bao nhiêu và vì sao?

In [8]:
bang_giam = pd.DataFrame({
    "clean": [res["clean"]["acc"], res["clean"]["macro_f1"], ret["bm25_clean"]["recall@1"], ret["bm25_clean"]["mrr@10"], ret["bm25_clean"]["ndcg@10"]],
    "noisy": [res["noisy"]["acc"], res["noisy"]["macro_f1"], ret["bm25_noisy"]["recall@1"], ret["bm25_noisy"]["mrr@10"], ret["bm25_noisy"]["ndcg@10"]],
}, index=["TF-IDF+LR acc", "TF-IDF+LR macro-F1", "BM25 recall@1", "BM25 mrr@10", "BM25 ndcg@10"])
bang_giam["giảm tuyệt đối"] = bang_giam.clean - bang_giam.noisy
bang_giam["giảm tương đối"] = bang_giam["giảm tuyệt đối"] / bang_giam.clean
print(bang_giam.round(3))

# Vì sao: đếm tỉ lệ từ "lạ" (OOV) = từ không có trong từ vựng của train + KB
tu_vung = {w for t in tr.text for w in tok(t)} | {w for t in kb.text for w in tok(t)}

def ti_le_oov(texts):
    tat_ca_tu = [w for t in texts for w in tok(t)]
    return np.mean([w not in tu_vung for w in tat_ca_tu])

tu_la = Counter(w for t in te_noisy.text for w in tok(t) if w not in tu_vung)
print(f"\nTỉ lệ từ lạ (OOV): clean {ti_le_oov(te.text):.2%} -> noisy {ti_le_oov(te_noisy.text):.2%}")
print("Từ lạ xuất hiện nhiều nhất trong tập noisy:", tu_la.most_common(8))

                    clean  noisy  giảm tuyệt đối  giảm tương đối
TF-IDF+LR acc       0.653  0.508           0.146           0.223
TF-IDF+LR macro-F1  0.634  0.487           0.147           0.232
BM25 recall@1       0.727  0.644           0.083           0.115
BM25 mrr@10         0.804  0.750           0.054           0.068
BM25 ndcg@10        0.844  0.801           0.043           0.050

Tỉ lệ từ lạ (OOV): clean 2.16% -> noisy 6.74%
Từ lạ xuất hiện nhiều nhất trong tập noisy: [('register', 60), ('submit', 58), ('tuition', 48), ('document', 44), ('scholarship', 36), ('graduation', 24), ('grade', 22), ('fee', 22)]


**Giải thích B3 (số ở bảng trên).** Sang tập nhiễu, TF-IDF + LR giảm accuracy từ 0,653 xuống 0,508 (−0,146, tức −22 %), macro-F1 từ 0,634 xuống 0,487. BM25 giảm Recall@1 từ 0,727 xuống 0,644 (−0,083) và MRR@10 từ 0,804 xuống 0,750.

**Vì sao:** `code_mix` thay đúng những từ khóa mang nhiều thông tin nhất (*học phí, đăng ký, nộp, hồ sơ, học bổng, tốt nghiệp…*) bằng từ tiếng Anh (*tuition, register, submit, document…*). Các từ này chưa từng có trong train và KB – tỉ lệ từ lạ tăng từ 2,2 % lên 6,7 %.
- Với **TF-IDF + LR**: từ lạ không có cột đặc trưng nên bị bỏ qua, mô hình mất đúng dấu hiệu quyết định chủ đề.
- Với **BM25**: từ lạ không có `idf` nên không được cộng điểm; đoạn đúng chỉ còn được chấm bằng các từ chung chung (*bao, nhiêu, thì, ở…*) nên dễ tụt hạng.

BM25 giảm ít hơn bộ phân loại vì câu hỏi vẫn còn các từ khác trùng với tên thủ tục. Cả hai baseline đều chỉ **khớp mặt chữ**, không hiểu `tuition` = *học phí*; đây là chỗ embedding đa ngữ ở Phần C có lợi thế.

## Phần C – Embedding, dense retrieval, hybrid
### C1. Dense retrieval với `embed_model` (cosine trên vector đã chuẩn hóa)

In [9]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(card["embed_model"], device=DEVICE)    # paraphrase-multilingual-MiniLM-L12-v2 (không cần tiền tố)

def embed(texts):
    """Câu -> vector 384 chiều đã chuẩn hóa độ dài 1 (để cosine = tích vô hướng)."""
    return embedder.encode([str(t) for t in texts], normalize_embeddings=True, batch_size=64, show_progress_bar=False)

P_emb = embed(kb.text)                                   # vector của 80 đoạn quy định
print("Ma trận embedding của KB:", P_emb.shape)

def dense_matrix(texts):
    return embed(texts) @ P_emb.T                        # [số câu hỏi x 80] = cosine giữa câu hỏi và từng đoạn

D_dev, D_rq, D_rqn = dense_matrix(dev.text), dense_matrix(rq.text), dense_matrix(rq_noisy.text)
ret["dense_clean"], ret["dense_noisy"] = eval_matrix(D_rq, rq), eval_matrix(D_rqn, rq_noisy)
print(pd.DataFrame({k: ret[k] for k in ["bm25_clean", "dense_clean", "bm25_noisy", "dense_noisy"]}).T.round(3))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Ma trận embedding của KB: (80, 384)
             recall@1  recall@3  recall@5  recall@10  mrr@10  ndcg@10
bm25_clean      0.727     0.841     0.894      0.970   0.804    0.844
dense_clean     0.576     0.758     0.818      0.909   0.674    0.730
bm25_noisy      0.644     0.818     0.894      0.962   0.750    0.801
dense_noisy     0.545     0.712     0.795      0.909   0.649    0.711


**Nhận xét C1.** Trên dữ liệu này dense **thấp hơn** BM25 (MRR@10 sạch 0,674 so với 0,804) vì câu hỏi thường nhắc đúng tên thủ tục có trong đoạn quy định nên khớp từ rất hiệu quả, còn mô hình embedding nhỏ dùng chung cho nhiều ngôn ngữ chưa được huấn luyện riêng cho miền này. Bù lại dense **ít bị ảnh hưởng bởi nhiễu** hơn: MRR@10 chỉ giảm 0,025 (BM25 giảm 0,054) vì embedding đa ngữ đặt `tuition` gần *học phí*.

### C2. Hybrid: `score = α·dense + (1−α)·bm25` (min-max theo từng truy vấn), chọn α trên **dev**

In [10]:
from sklearn.preprocessing import minmax_scale

def hybrid_scores(D, B, alpha):
    """minmax_scale(axis=1): đưa điểm của TỪNG câu hỏi (từng hàng) về [0, 1] để hai thang điểm cộng được với nhau."""
    return alpha * minmax_scale(D, axis=1) + (1 - alpha) * minmax_scale(B, axis=1)

# Quét α = 0.0, 0.1, ..., 1.0 trên DEV (không đụng tới test)
ALPHAS = [round(a, 1) for a in np.arange(0, 1.01, 0.1)]
sweep_dev = pd.DataFrame({a: eval_matrix(hybrid_scores(D_dev, B_dev, a), dev) for a in ALPHAS}).T
sweep_dev.index.name = "alpha"
print("Quét α trên DEV:")
print(sweep_dev.round(3))

alpha_best = ALPHAS[int(np.argmax(sweep_dev["mrr@10"].values))]      # tiêu chí chọn: MRR@10 cao nhất trên dev
print("\nα tốt nhất trên dev =", alpha_best)

# Chốt α rồi mới báo cáo trên retr_test
ret["hybrid_clean"] = eval_matrix(hybrid_scores(D_rq, B_rq, alpha_best), rq)
ret["hybrid_noisy"] = eval_matrix(hybrid_scores(D_rqn, B_rqn, alpha_best), rq_noisy)
print("\nBáo cáo trên retr_test:")
print(pd.DataFrame(ret).T.loc[["bm25_clean", "dense_clean", "hybrid_clean", "bm25_noisy", "dense_noisy", "hybrid_noisy"]].round(3))

Quét α trên DEV:
       recall@1  recall@3  recall@5  recall@10  mrr@10  ndcg@10
alpha                                                          
0.0       0.629     0.733     0.784      0.879   0.699    0.741
0.1       0.629     0.724     0.819      0.879   0.705    0.747
0.2       0.621     0.759     0.836      0.905   0.711    0.758
0.3       0.629     0.767     0.845      0.922   0.721    0.769
0.4       0.638     0.767     0.845      0.948   0.729    0.781
0.5       0.629     0.784     0.871      0.966   0.732    0.788
0.6       0.629     0.810     0.888      0.966   0.737    0.792
0.7       0.629     0.802     0.862      0.966   0.738    0.793
0.8       0.612     0.784     0.836      0.948   0.718    0.773
0.9       0.560     0.750     0.793      0.914   0.673    0.730
1.0       0.491     0.681     0.767      0.862   0.608    0.669

α tốt nhất trên dev = 0.7

Báo cáo trên retr_test:
              recall@1  recall@3  recall@5  recall@10  mrr@10  ndcg@10
bm25_clean       0.727     0

### C3. Phân loại bằng embedding đóng băng + Logistic Regression (so với B1)

In [11]:
# "Đóng băng": không huấn luyện mô hình embedding, chỉ huấn luyện Logistic Regression trên vector của nó
E_tr, E_te, E_ten = embed(tr.text), embed(te.text), embed(te_noisy.text)
clf_e = LogisticRegression(max_iter=2000, random_state=SEED).fit(E_tr, tr.topic)

pred_emb = {"clean": clf_e.predict(E_te), "noisy": clf_e.predict(E_ten)}
res_emb = {"clean": cham_phan_loai(te.topic, pred_emb["clean"]), "noisy": cham_phan_loai(te_noisy.topic, pred_emb["noisy"])}

so_sanh = pd.DataFrame({
    "TF-IDF+LR (B1)":    {f"{tap} {m}": res[tap][m] for tap in ("clean", "noisy") for m in ("acc", "macro_f1")},
    "Embedding+LR (C3)": {f"{tap} {m}": res_emb[tap][m] for tap in ("clean", "noisy") for m in ("acc", "macro_f1")},
})
so_sanh["chênh lệch (C3 − B1)"] = so_sanh["Embedding+LR (C3)"] - so_sanh["TF-IDF+LR (B1)"]
print(so_sanh.round(4))

                TF-IDF+LR (B1)  Embedding+LR (C3)  chênh lệch (C3 − B1)
clean acc               0.6534             0.6364               -0.0170
clean macro_f1          0.6344             0.6204               -0.0140
noisy acc               0.5076             0.6136                0.1061
noisy macro_f1          0.4871             0.5926                0.1055


**Nhận xét C3.** Trên tập **sạch** hai cách gần ngang nhau (accuracy 0,636 so với 0,653). Trên tập **nhiễu** embedding hơn rõ (0,614 so với 0,508, tức +0,106): mô hình đa ngữ hiểu `tuition ≈ học phí`, `register ≈ đăng ký` nên từ tiếng Anh vẫn mang nghĩa, còn TF-IDF coi đó là từ lạ và bỏ qua. Embedding chỉ mất 0,02 accuracy khi có nhiễu, TF-IDF mất 0,146.

### C4. [Nâng cao] LoRA fine-tune `clf_backbone` với cấu hình `lora` của thẻ
Cell chỉ chạy khi có GPU (Colab T4). Siêu tham số huấn luyện (lr, số epoch, batch) đặt cố định từ trước, không chỉnh theo test.

In [12]:
RUN_LORA = (DEVICE == "cuda")          # đặt True để ép chạy trên CPU (rất chậm)
lora_info = {"trainable_params": None, "note": "chưa chạy (không có GPU)"}

if RUN_LORA:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from peft import LoraConfig, get_peft_model, TaskType

    label2id = {t: i for i, t in enumerate(TOPICS)}
    id2label = {i: t for t, i in label2id.items()}
    tok_lora = AutoTokenizer.from_pretrained(card["clf_backbone"])
    base = AutoModelForSequenceClassification.from_pretrained(card["clf_backbone"], num_labels=len(TOPICS))

    # Gắn LoRA theo đúng thẻ: r, alpha, dropout, target_modules = [query, value]
    cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=card["lora"]["r"], lora_alpha=card["lora"]["alpha"],
                     lora_dropout=card["lora"]["dropout"], target_modules=card["lora"]["target_modules"])
    lora_model = get_peft_model(base, cfg).to(DEVICE)
    lora_model.print_trainable_parameters()                       # <- con số đề yêu cầu ghi lại

    n_trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
    n_lora = sum(p.numel() for ten, p in lora_model.named_parameters() if "lora_" in ten)     # chỉ các ma trận A, B
    print(f"trainable = {n_trainable:,} | trong đó riêng ma trận LoRA (A, B) = {n_lora:,} (đối chiếu E1); phần còn lại là đầu phân loại")

    # Huấn luyện: AdamW, mini-batch
    LORA_LR, LORA_EPOCHS, LORA_BS, MAX_LEN = 2e-4, 4, 16, 64
    y_tr = np.array([label2id[t] for t in tr.topic])
    opt = torch.optim.AdamW([p for p in lora_model.parameters() if p.requires_grad], lr=LORA_LR)
    for ep in range(LORA_EPOCHS):
        lora_model.train()
        thu_tu = np.random.RandomState(SEED + ep).permutation(len(tr))      # xáo trộn dữ liệu theo seed
        tong_loss = 0.0
        for i in range(0, len(tr), LORA_BS):
            idx = thu_tu[i:i + LORA_BS]
            batch = tok_lora(list(tr.text.values[idx]), padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
            out = lora_model(**batch, labels=torch.tensor(y_tr[idx]).to(DEVICE))
            out.loss.backward()
            opt.step()
            opt.zero_grad()
            tong_loss += out.loss.item() * len(idx)
        print(f"epoch {ep + 1}/{LORA_EPOCHS} | loss train = {tong_loss / len(tr):.4f}")

    def lora_predict(texts, bs=64):
        lora_model.eval()
        nhan = []
        with torch.no_grad():
            for i in range(0, len(texts), bs):
                batch = tok_lora(list(texts[i:i + bs]), padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
                nhan += lora_model(**batch).logits.argmax(-1).cpu().tolist()
        return [id2label[i] for i in nhan]

    res_lora = {"clean": cham_phan_loai(te.topic, lora_predict(te.text.values)),
                "noisy": cham_phan_loai(te_noisy.topic, lora_predict(te_noisy.text.values))}
    lora_info = {"trainable_params": n_trainable, "lora_only_params": n_lora, **res_lora}
    print(pd.DataFrame(res_lora).T.round(4))
else:
    print("Không có GPU -> bỏ qua C4 (phần nâng cao). Số tham số LoRA được tính tay ở E1.")

Không có GPU -> bỏ qua C4 (phần nâng cao). Số tham số LoRA được tính tay ở E1.


## Phần D – LLM: prompting và RAG
### D1. Zero-shot và few-shot trên `llm_eval_30.csv` (temperature = 0)
**Mô hình:** `Qwen/Qwen2.5-1.5B-Instruct` (LLM mã nguồn mở, chạy được trên CPU). **Phiên bản:** 2.5, bản 1.5B-Instruct, tải từ Hugging Face bằng `pipeline("text-generation")` của thư viện `transformers`. Giải mã **greedy** (`do_sample=False`), tương đương temperature = 0 nên kết quả lặp lại được.

In [13]:
from transformers import pipeline

LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
LLM_VERSION = "2.5 (1.5B-Instruct)"                    # phiên bản và kích cỡ nằm ngay trong tên mô hình
llm = pipeline("text-generation", model=LLM_NAME, device=DEVICE,
               dtype=torch.float16 if DEVICE == "cuda" else torch.float32)
print("Mô hình:", LLM_NAME, "| phiên bản:", LLM_VERSION)

def llm_generate(messages, max_new_tokens=24):
    """Đưa hội thoại (system / user / assistant) vào LLM, trả về phần văn bản mô hình sinh thêm.
    pipeline tự làm 4 bước: áp khuôn chat -> tách token -> sinh -> giải mã phần mới. do_sample=False = greedy = temperature 0."""
    out = llm(messages, max_new_tokens=max_new_tokens, do_sample=False, return_full_text=False)
    return out[0]["generated_text"].strip()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Mô hình: Qwen/Qwen2.5-1.5B-Instruct | phiên bản: 2.5 (1.5B-Instruct)


In [14]:
TOPIC_DESC = {   # mô tả 8 chủ đề (theo data/topics.json của giảng viên)
    "dang_ky_hoc_phan": "Đăng ký học phần & kế hoạch học tập", "hoc_phi": "Học phí",
    "hoc_bong": "Học bổng & trợ cấp", "thi_diem": "Thi cử & điểm số",
    "tot_nghiep": "Tốt nghiệp & văn bằng", "ky_tuc_xa": "Ký túc xá",
    "thu_vien_cntt": "Thư viện & dịch vụ CNTT", "ctsv": "Công tác sinh viên & giấy tờ"}

SYSTEM_PROMPT = ("Bạn là bộ phân loại câu hỏi của sinh viên về thủ tục đại học. "
                 "Hãy chọn đúng MỘT mã chủ đề phù hợp nhất trong danh sách sau và chỉ trả lời bằng mã đó, không giải thích.\n"
                 "Danh sách chủ đề:\n" + "\n".join(f"- {ma}: {mo_ta}" for ma, mo_ta in TOPIC_DESC.items()))

def user_turn(text):
    return f"Câu hỏi: {text}\nMã chủ đề:"

# ---- Chọn ví dụ few-shot: tổng cộng k = fewshot_k ví dụ, mỗi ví dụ một chủ đề khác nhau, ngẫu nhiên theo SEED ----
K_SHOT = card["fewshot_k"]
pool = fewshot.merge(kb[["passage_id", "topic"]], left_on="gold_passage_id", right_on="passage_id")   # thêm cột topic cho kho ví dụ
rng = np.random.RandomState(SEED)
shot_topics = rng.choice(TOPICS, K_SHOT, replace=False)                                               # k chủ đề khác nhau
shots = [pool[pool.topic == t].sample(1, random_state=int(rng.randint(0, 10**6))).iloc[0] for t in shot_topics]
print(f"{K_SHOT} ví dụ few-shot:")
for s in shots:
    print(f"  [{s.topic}] {s.text}")

def build_messages(text, use_shots):
    """Zero-shot: system + câu hỏi. Few-shot: chèn thêm k cặp (câu hỏi ví dụ -> mã chủ đề đúng) vào trước."""
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    if use_shots:
        for s in shots:
            msgs.append({"role": "user", "content": user_turn(s.text)})
            msgs.append({"role": "assistant", "content": s.topic})
    msgs.append({"role": "user", "content": user_turn(text)})
    return msgs

print("\n=========== PROMPT ĐẦY ĐỦ (few-shot, câu đầu tiên của tập đánh giá) ===========")
print(llm.tokenizer.apply_chat_template(build_messages(llm_eval.text.iloc[0], True), tokenize=False, add_generation_prompt=True))

6 ví dụ few-shot:
  [hoc_bong] Bạn ơi, thấy kết quả xét học bổng chưa đúng nên thắc mắc thì phải đến đâu để làm thủ tục?
  [thi_diem] Có deadline cho việc tra cứu lịch thi không, là bao giờ?
  [ky_tuc_xa] Cho hỏi, muốn ở nội trú trong trường thì thời gian nộp hồ sơ ra sao?
  [ctsv] Bạn ơi, cần giấy gửi cho địa phương để hoãn đi nghĩa vụ thì có hạn chót không?
  [thu_vien_cntt] Mình cần địa chỉ thư điện tử của trường thì có phải đóng phí không?
  [hoc_phi] Bạn ơi, nhà thuộc hộ nghèo muốn được giảm tiền học thì phải đến đâu để làm thủ tục?

=========== PROMPT ĐẦY ĐỦ (few-shot, câu đầu tiên của tập đánh giá) ===========
<|im_start|>system
Bạn là bộ phân loại câu hỏi của sinh viên về thủ tục đại học. Hãy chọn đúng MỘT mã chủ đề phù hợp nhất trong danh sách sau và chỉ trả lời bằng mã đó, không giải thích.
Danh sách chủ đề:
- dang_ky_hoc_phan: Đăng ký học phần & kế hoạch học tập
- hoc_phi: Học phí
- hoc_bong: Học bổng & trợ cấp
- thi_diem: Thi cử & điểm số
- tot_nghiep: Tốt nghiệp & văn bằng

In [15]:
def parse_label(output):
    """Lấy mã chủ đề xuất hiện sớm nhất trong câu trả lời của LLM; không thấy mã nào -> 'invalid'."""
    tim_thay = [(output.find(t), t) for t in TOPICS if t in output]
    return min(tim_thay)[1] if tim_thay else "invalid"

def run_prompting(use_shots):
    tra_loi = [llm_generate(build_messages(t, use_shots)) for t in llm_eval.text]
    return pd.DataFrame({"qid": llm_eval.qid, "text": llm_eval.text, "gold": llm_eval.topic,
                         "raw": tra_loi, "pred": [parse_label(o) for o in tra_loi]})

zs = run_prompting(use_shots=False)        # zero-shot
ks = run_prompting(use_shots=True)         # k-shot
zero_shot_acc = float((zs.pred == zs.gold).mean())
k_shot_acc = float((ks.pred == ks.gold).mean())
print(f"Zero-shot : {zero_shot_acc:.3f} ({(zs.pred == zs.gold).sum()}/{len(zs)})")
print(f"{K_SHOT}-shot    : {k_shot_acc:.3f} ({(ks.pred == ks.gold).sum()}/{len(ks)})")

print(f"\n3 câu sai tiêu biểu ({K_SHOT}-shot):")
print(ks[ks.pred != ks.gold].head(3)[["qid", "text", "gold", "pred"]].to_string(index=False))

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=24) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=24) a

Zero-shot : 0.400 (12/30)
6-shot    : 0.533 (16/30)

3 câu sai tiêu biểu (6-shot):
    qid                                                            text          gold    pred
Q074-21             Em muốn đóng bảo hiểm y tế, vậy tốn bao nhiêu tiền?          ctsv hoc_phi
Q066-01       Muốn cấp lại mật khẩu cổng thông tin thì thực hiện ở đâu? thu_vien_cntt    ctsv
Q078-22 Mình muốn vào câu lạc bộ của trường thì có phải đóng phí không?          ctsv hoc_phi


**Nhận xét D1.** Few-shot (6 ví dụ) tốt hơn zero-shot: 0,533 so với 0,400 trên 30 câu, vì ví dụ cho mô hình thấy đúng định dạng trả lời (chỉ một mã) và gắn vài kiểu câu hỏi với nhãn. Mô hình chỉ 1,5 tỉ tham số nên kết quả còn thấp hơn hai bộ phân loại có huấn luyện ở B1/C3.

**Cách chọn ví dụ:** tổng cộng k = `fewshot_k` = 6 ví dụ lấy từ `fewshot_pool.csv` (các mục train), mỗi ví dụ thuộc một chủ đề khác nhau; chủ đề và ví dụ được chọn ngẫu nhiên theo `SEED`. Vì k = 6 < 8 nên có 2 chủ đề không có ví dụ.

**3 câu sai tiêu biểu (6-shot) và lý do:**
1. *"Em muốn đóng bảo hiểm y tế, vậy tốn bao nhiêu tiền?"* – đúng `ctsv`, mô hình đoán `hoc_phi`: cụm *"đóng… bao nhiêu tiền"* kéo về học phí; câu không nói rõ đây là thủ tục của Phòng Công tác sinh viên (thiếu ngữ cảnh).
2. *"Muốn cấp lại mật khẩu cổng thông tin thì thực hiện ở đâu?"* – đúng `thu_vien_cntt`, đoán `ctsv`: *"cấp lại"* giống các thủ tục giấy tờ của công tác sinh viên (cấp lại thẻ sinh viên) → từ khóa gây nhầm.
3. *"Mình muốn vào câu lạc bộ của trường thì có phải đóng phí không?"* – đúng `ctsv`, đoán `hoc_phi`: từ *"đóng phí"* gây nhầm; mô tả chủ đề `ctsv` trong prompt không nhắc tới câu lạc bộ.

### D2. RAG: truy hồi top-`rag_top_k` → ghép ngữ cảnh → trích xuất câu trả lời
Mỗi đoạn quy định đều có 3 thông tin: **nơi** tiếp nhận, **hạn** (thời hạn), **phí** (lệ phí). Quy trình cho một câu hỏi:
1. **Truy hồi** top-K đoạn bằng hybrid (α đã chốt trên dev ở C2), K = `rag_top_k`.
2. **Xác định câu hỏi đang hỏi gì** (nơi / hạn / phí) bằng một bộ phân loại TF-IDF + LR huấn luyện trên cột `field` của train.
3. **Trích xuất** thông tin đó trong từng đoạn top-K (bằng regex theo khuôn câu của KB) và ghép lại thành câu trả lời.

`answer_hit_rate` = tỉ lệ câu trả lời có chứa `gold_answer` (sau `normalize`), đo trên toàn bộ 132 câu của `retr_test` (đề yêu cầu ≥ 60).

In [16]:
RAG_K = card["rag_top_k"]            # số đoạn đưa vào ngữ cảnh (giảng viên có thể yêu cầu đổi số này rồi chạy lại)

# Bộ phân loại "câu hỏi đang hỏi nơi / hạn / phí?" – huấn luyện trên cột field của train
fvec = TfidfVectorizer(preprocessor=normalize, ngram_range=(1, 2), sublinear_tf=True)
fclf = LogisticRegression(max_iter=2000, random_state=SEED).fit(fvec.fit_transform(tr.text), tr["field"])

# KB chỉ có hai khuôn câu:  "Đơn vị tiếp nhận: X. Thời hạn: Y. Lệ phí: Z."
#                     và:  "... có thể thực hiện tại X. Thời hạn thực hiện: Y. Mức lệ phí: Z."
SLOT_PATTERNS = {
    "noi": r"(?:Đơn vị tiếp nhận:|có thể thực hiện tại)\s*(.+?)\.\s*Thời hạn",
    "han": r"Thời hạn(?: thực hiện)?:\s*(.+?)\.\s*(?:Lệ phí|Mức lệ phí)",
    "phi": r"(?:Lệ phí|Mức lệ phí):\s*(.+?)\.?\s*$",
}

def extract_slot(passage_text, field):
    """Lấy giá trị nơi / hạn / phí trong một đoạn quy định."""
    m = re.search(SLOT_PATTERNS[field], passage_text)
    return m.group(1).strip().rstrip(".") if m else None

def top_k_passages(hybrid_row, k):
    return np.argsort(-hybrid_row, kind="stable")[:k]             # chỉ số của k đoạn điểm cao nhất

def extract_answer(top_idx, field):
    """Ghép ngữ cảnh: trích giá trị `field` trong từng đoạn top-K, nối lại (bỏ giá trị trùng)."""
    gia_tri = [extract_slot(kb.text.values[j], field) for j in top_idx]
    return " / ".join(dict.fromkeys(v for v in gia_tri if v))

H_rq = hybrid_scores(D_rq, B_rq, alpha_best)
field_rq = fclf.predict(fvec.transform(rq.text))

rag_rows = []
for i in range(len(rq)):
    top_idx = top_k_passages(H_rq[i], RAG_K)
    answer = extract_answer(top_idx, field_rq[i])
    rag_rows.append({"text": rq.text.iloc[i], "top_ids": [kb.passage_id.values[j] for j in top_idx],
                     "answer": answer, "gold": rq.gold_answer.iloc[i],
                     "hit": normalize(rq.gold_answer.iloc[i]) in normalize(answer)})
rag = pd.DataFrame(rag_rows)

answer_hit_rate = float(rag.hit.mean())
print(f"answer_hit_rate = {answer_hit_rate:.3f} ({rag.hit.sum()}/{len(rag)} câu, top-{RAG_K})")
print("\nVài câu trả lời đúng:")
print(rag[rag.hit].sample(3, random_state=SEED)[["text", "answer", "gold"]].to_string(index=False))
print("\nVài câu trả lời sai:")
print(rag[~rag.hit].head(3)[["text", "top_ids", "answer", "gold"]].to_string(index=False))

answer_hit_rate = 0.879 (116/132 câu, top-2)

Vài câu trả lời đúng:
                                                                     text                                                  answer                     gold
                        Mình phải xin trợ cấp xã hội trước thời điểm nào? 45 ngày đầu học kỳ / 10 ngày sau khi kết thúc hoạt động       45 ngày đầu học kỳ
             Mình cần xin trợ cấp xã hội, phải liên hệ phòng ban nào vậy?                  Phòng Công tác sinh viên / Đoàn trường Phòng Công tác sinh viên
Cho hỏi, mang xe máy vào ký túc xá để gửi thì thời gian nộp hồ sơ ra sao?          trong tuần đầu vào ở / 5 ngày kể từ ngày vào ở     trong tuần đầu vào ở

Vài câu trả lời sai:
                                                     text      top_ids                                     answer                                 gold
 Mình lỡ rớt môn muốn học lại thì có phải đóng phí không? [P033, P016]      100.000 đồng mỗi môn / không tính lãi bằng học phí theo số tín 

### D3. Từ chối trả lời bằng ngưỡng điểm truy hồi
- **Điểm dùng để đặt ngưỡng:** cosine cao nhất giữa câu hỏi và 80 đoạn (dense top-1). Không dùng điểm hybrid vì sau min-max đoạn đứng đầu luôn có điểm 1,0.
- **Chọn ngưỡng τ chỉ trên dev** (không dùng `ood_10` hay test): τ = phân vị thứ 5 của cosine top-1 trên các câu hợp lệ của dev, nghĩa là chấp nhận từ chối nhầm khoảng 5 % câu hợp lệ.
- Câu hỏi có cosine top-1 < τ → hệ thống trả lời *"Không tìm thấy thông tin trong quy định."*

In [17]:
ABSTAIN_MSG = "Không tìm thấy thông tin trong quy định."

tau = float(np.percentile(D_dev.max(axis=1), 5))            # ngưỡng chốt trên dev

top1_ood = dense_matrix(ood.text).max(axis=1)               # 10 câu NGOÀI phạm vi
top1_rq = D_rq.max(axis=1)                                  # 132 câu HỢP LỆ của retr_test
abstain_rate_ood = float((top1_ood < tau).mean())           # tỉ lệ câu ngoài phạm vi bị từ chối (càng cao càng tốt)
false_abstain_rate = float((top1_rq < tau).mean())          # tỉ lệ câu hợp lệ bị từ chối nhầm (càng thấp càng tốt)

print(f"Ngưỡng τ = {tau:.4f}")
print(f"abstain_rate_ood   = {abstain_rate_ood:.3f} ({(top1_ood < tau).sum()}/{len(ood)} câu ngoài phạm vi bị từ chối)")
print(f"false_abstain_rate = {false_abstain_rate:.3f} ({(top1_rq < tau).sum()}/{len(rq)} câu hợp lệ bị từ chối nhầm)")
print("\nTừng câu ngoài phạm vi:")
print(ood.assign(cosine_top1=top1_ood.round(3), tu_choi=top1_ood < tau)[["kind", "text", "cosine_top1", "tu_choi"]].to_string(index=False))

def answer_question(q):
    """Hệ thống hoàn chỉnh cho MỘT câu hỏi: truy hồi -> từ chối nếu cosine top-1 < τ -> trích xuất câu trả lời."""
    d, b = dense_matrix([q]), bm25_matrix([q])
    if d.max() < tau:
        return ABSTAIN_MSG
    top_idx = top_k_passages(hybrid_scores(d, b, alpha_best)[0], RAG_K)
    field = fclf.predict(fvec.transform([q]))[0]
    return extract_answer(top_idx, field)

for q in ["Mình cần phúc khảo bài thi, phải liên hệ phòng ban nào vậy?",      # câu hợp lệ (có trong retr_test)
          "Giá vàng SJC hôm nay bao nhiêu một lượng?"]:                          # câu ngoài phạm vi (có trong ood_10)
    print(f"\nHỏi: {q}\nĐáp: {answer_question(q)}")

Ngưỡng τ = 0.4517
abstain_rate_ood   = 0.300 (3/10 câu ngoài phạm vi bị từ chối)
false_abstain_rate = 0.068 (9/132 câu hợp lệ bị từ chối nhầm)

Từng câu ngoài phạm vi:
       kind                                                                      text  cosine_top1  tu_choi
totally_off                                 Giá vàng SJC hôm nay bao nhiêu một lượng?        0.239     True
 near_topic                          Thủ tục đăng ký dạy kèm cho sinh viên khóa dưới?        0.724    False
 near_topic     Sinh viên năm cuối có được miễn thi ngoại ngữ nếu có IELTS 8.0 không?        0.583    False
totally_off                                       Hôm nay thời tiết ở Hà Nội thế nào?        0.305     True
totally_off                                            Ai vô địch World Cup 2022 vậy?        0.216     True
 near_topic     Muốn xin học bổng toàn phần du học Mỹ thì nộp hồ sơ ở đâu và hạn nào?        0.644    False
 near_topic              Trường có tổ chức khám sức khỏe định kỳ cho sinh vi

**Nhận xét D3.** Hệ thống từ chối được 3/10 câu ngoài phạm vi và từ chối nhầm 9/132 câu hợp lệ (6,8 %). Ba câu bị từ chối đều thuộc loại `totally_off` (giá vàng, thời tiết, World Cup – cosine chỉ 0,22–0,31). Bảy câu còn lại thuộc loại `near_topic`: vẫn nói về trường học (học bổng du học, khám sức khỏe, điểm chuẩn…) nên cosine cao 0,51–0,72, ngang với câu hợp lệ, và một ngưỡng đơn giản không tách được. Nếu nâng τ để bắt thêm các câu này thì số câu hợp lệ bị từ chối nhầm cũng tăng theo – đây là sự đánh đổi của phương pháp ngưỡng.

## Phần E – Post-training: "tính tay" rồi kiểm bằng code
Mỗi thành viên làm tay E1–E3 theo **thẻ riêng** của mình (ảnh chụp lời giải tay nộp kèm). Dưới đây là lời giải tay theo thẻ nhóm trưởng (24DH190314); cell code bên dưới kiểm lại lời giải của **cả ba** thành viên.

Thẻ 24DH190314: `clf_backbone = xlm-roberta-base` (12 lớp, hidden d = 768), `r = 8`; DPO `β = 0,5`, `logπ(chosen) = −18,5`, `logπ(rejected) = −42,8`, `logref(chosen) = −24,8`, `logref(rejected) = −32,3`; `toy_ranking = [0,1,1,0,0,0,0,0]`.

**E1 – Số tham số LoRA.** LoRA giữ nguyên ma trận gốc `W (d×d)` và chỉ học phần cập nhật `ΔW = B·A`, với `A` cỡ `r×d` và `B` cỡ `d×r`.
- 1 ma trận: `r·d + d·r = 2·r·d = 2·8·768 = 12.288`
- 1 lớp có 2 ma trận (query, value): `2 × 12.288 = 24.576`
- 12 lớp: `12 × 24.576 = 294.912` tham số
- So sánh: một ma trận đầy đủ `d×d = 768² = 589.824`. Một ma trận LoRA chỉ bằng `12.288 / 589.824 = 1/48 ≈ 2,08 %`; toàn bộ LoRA của 12 lớp (294.912) mới bằng **một nửa** của một ma trận đầy đủ.

**E2 – Loss DPO.** `h = β·[(logπ_chosen − logref_chosen) − (logπ_rejected − logref_rejected)]`, `loss = −ln σ(h)`
- chosen: `−18,5 − (−24,8) = +6,3` ; rejected: `−42,8 − (−32,3) = −10,5`
- `h = 0,5 · (6,3 − (−10,5)) = 0,5 · 16,8 = 8,4`
- `loss = −ln σ(8,4) = ln(1 + e^(−8,4)) ≈ 2,248·10⁻⁴`
- **Dấu của margin:** `h > 0` nghĩa là so với mô hình tham chiếu, mô hình đang huấn luyện đã *tăng* xác suất câu chosen (+6,3) và *giảm* xác suất câu rejected (−10,5) → đang ưa câu chosen đúng như mong muốn, nên loss gần 0. Nếu `h < 0` thì mô hình đang ưa câu rejected hơn, loss lớn và gradient kéo mạnh về phía chosen.

**E3 – Metric xếp hạng.** `toy_ranking = [0,1,1,0,0,0,0,0]`: có 2 kết quả liên quan, nằm ở hạng 2 và hạng 3; `k = 5`.
- `P@5 = 2/5 = 0,4` ; `R@5 = 2/2 = 1,0` ; `RR = 1/2 = 0,5` (kết quả liên quan đầu tiên ở hạng 2)
- `DCG@5 = 1/log₂(3) + 1/log₂(4) = 0,6309 + 0,5 = 1,1309`
- `IDCG@5 = 1/log₂(2) + 1/log₂(3) = 1 + 0,6309 = 1,6309` (xếp lý tưởng: 2 kết quả liên quan ở hạng 1 và 2)
- `nDCG@5 = 1,1309 / 1,6309 ≈ 0,6934`

In [18]:
def lora_trainable(layers, hidden, r, n_targets=2):
    """E1: số tham số LoRA khi gắn vào n_targets ma trận (hidden x hidden) ở mỗi lớp."""
    mot_ma_tran = r * hidden + hidden * r            # A (r x d) + B (d x r) = 2·r·d
    return layers * n_targets * mot_ma_tran

def dpo_margin(beta, lp_pi_w, lp_pi_l, lp_ref_w, lp_ref_l):
    """E2: margin h = beta * ((pi_w - ref_w) - (pi_l - ref_l));  w = chosen, l = rejected."""
    return beta * ((lp_pi_w - lp_ref_w) - (lp_pi_l - lp_ref_l))

def dpo_loss(beta, lp_pi_w, lp_pi_l, lp_ref_w, lp_ref_l):
    """E2: loss = -log sigmoid(h) = log(1 + e^(-h))."""
    h = dpo_margin(beta, lp_pi_w, lp_pi_l, lp_ref_w, lp_ref_l)
    return math.log(1 + math.exp(-h))

def toy_metrics(rels, k=5):
    """E3: P@k, R@k, RR, nDCG@k cho danh sách nhị phân `rels` (1 = liên quan), phần tử đầu là hạng 1."""
    so_lien_quan = sum(rels)
    trung_top_k = sum(rels[:k])
    rr = next((1 / (i + 1) for i, r in enumerate(rels) if r), 0.0)            # 1 / hạng của kết quả liên quan đầu tiên
    dcg = sum(r / math.log2(i + 2) for i, r in enumerate(rels[:k]))           # hạng (i+1) -> chia log2(i+2)
    idcg = sum(1 / math.log2(i + 2) for i in range(min(so_lien_quan, k)))     # lý tưởng: mọi kết quả liên quan lên đầu
    return {f"P@{k}": trung_top_k / k, f"R@{k}": trung_top_k / so_lien_quan, "RR": rr, f"nDCG@{k}": dcg / idcg}

# (số lớp, hidden) của từng backbone – cả hai đều là cỡ "base"
BACKBONE = {"xlm-roberta-base": (12, 768), "vinai/phobert-base-v2": (12, 768)}

# Kết quả TÍNH TAY của từng thành viên (chép từ lời giải trên giấy)
LOI_GIAI_TAY = {
    "24DH190314": {"lora": 294912, "h": 8.4,  "loss": 2.248e-4, "P@5": 0.4, "R@5": 1.0, "RR": 0.5,  "nDCG@5": 0.6934},
    "24DH190309": {"lora": 294912, "h": 1.31, "loss": 0.2389,   "P@5": 0.4, "R@5": 1.0, "RR": 1.0,  "nDCG@5": 0.8503},
    "24DH190312": {"lora": 147456, "h": 1.8,  "loss": 0.1530,   "P@5": 0.2, "R@5": 0.5, "RR": 0.25, "nDCG@5": 0.2641},
}

from lab9_tools import assign                 # công cụ của giảng viên: tạo lại thẻ cá nhân từ MSSV

ket_qua_E = {}
for ho_ten, mssv, _, _ in NHOM:
    the = assign(mssv)                                             # thẻ riêng của thành viên
    layers, hidden = BACKBONE[the["clf_backbone"]]
    d = the["dpo"]
    so_dpo = (d["beta"], d["logp_pi_chosen"], d["logp_pi_rejected"], d["logp_ref_chosen"], d["logp_ref_rejected"])
    code_tinh = {"lora": lora_trainable(layers, hidden, the["lora"]["r"]),
                 "h": dpo_margin(*so_dpo), "loss": dpo_loss(*so_dpo), **toy_metrics(the["toy_ranking"], k=5)}

    tay = LOI_GIAI_TAY[mssv]                                       # so code với lời giải tay (sai số do làm tròn)
    assert code_tinh["lora"] == tay["lora"], f"E1 sai ({mssv})"
    assert abs(code_tinh["h"] - tay["h"]) < 1e-6 and abs(code_tinh["loss"] - tay["loss"]) < 5e-5, f"E2 sai ({mssv})"
    assert all(abs(code_tinh[m] - tay[m]) < 5e-5 for m in ("P@5", "R@5", "RR", "nDCG@5")), f"E3 sai ({mssv})"
    ket_qua_E[mssv] = {"ho_ten": ho_ten, "check_code": the["check_code"], "backbone": the["clf_backbone"], "r": the["lora"]["r"], **code_tinh}

assert ket_qua_E[MSSV]["check_code"] == card["check_code"]         # thẻ nhóm trưởng tạo lại khớp thẻ đang dùng
print("Code khớp lời giải tay của cả", len(NHOM), "thành viên:")
print(pd.DataFrame(ket_qua_E).T.to_string())

# So sánh E1 với một ma trận đầy đủ d x d (thẻ nhóm trưởng)
layers, hidden = BACKBONE[card["clf_backbone"]]
lora_total = ket_qua_E[MSSV]["lora"]
print(f"\nE1 (thẻ nhóm trưởng): 1 ma trận LoRA = {lora_trainable(1, hidden, card['lora']['r'], 1):,} | tổng query+value {layers} lớp = {lora_total:,} "
      f"| 1 ma trận đầy đủ d×d = {hidden * hidden:,} | tỉ lệ tổng LoRA / d×d = {lora_total / hidden**2:.2f}")

Code khớp lời giải tay của cả 3 thành viên:
                       ho_ten check_code               backbone  r    lora     h      loss  P@5  R@5    RR    nDCG@5
24DH190314  Nguyễn Hoàng Phúc   927D30B4       xlm-roberta-base  8  294912   8.4  0.000225  0.4  1.0   0.5  0.693426
24DH190309     Nguyễn Bảo Duy   CD4A20E5       xlm-roberta-base  8  294912  1.31  0.238875  0.4  1.0   1.0  0.850345
24DH190312           Lê Trần Nhật Huy   74695247  vinai/phobert-base-v2  4  147456   1.8  0.152978  0.2  0.5  0.25  0.264068

E1 (thẻ nhóm trưởng): 1 ma trận LoRA = 12,288 | tổng query+value 12 lớp = 294,912 | 1 ma trận đầy đủ d×d = 589,824 | tỉ lệ tổng LoRA / d×d = 0.50


**Lưu ý E1.** Nếu chạy C4, `print_trainable_parameters()` báo con số *lớn hơn* 294.912 vì ngoài các ma trận LoRA, bài toán phân loại còn huấn luyện thêm **đầu phân loại** (classifier). Riêng các ma trận A, B của LoRA đúng bằng 294.912.

**Dữ liệu `preference_pairs.jsonl` (minh họa cho DPO).** Mỗi dòng là một bộ (prompt, chosen, rejected): *chosen* là câu trả lời bám đúng ngữ cảnh, *rejected* là câu trả lời nghe hợp lý nhưng không có trong ngữ cảnh (kiểu "ảo giác"). DPO dùng các cặp như vậy để tăng xác suất câu chosen và giảm xác suất câu rejected; công thức loss chính là công thức đã tính tay ở E2. Nhóm chỉ thảo luận, không huấn luyện DPO.

In [19]:
pairs = [json.loads(dong) for dong in open(PACK / "preference_pairs.jsonl", encoding="utf-8")]
print("Số cặp:", len(pairs))
print(json.dumps(pairs[0], ensure_ascii=False, indent=1))

Số cặp: 80
{
 "prompt": "Ngữ cảnh: Hướng dẫn đăng ký học phần học kỳ chính. Đối tượng áp dụng: sinh viên đã hoàn tất học phí học kỳ trước. Đơn vị tiếp nhận: cổng thông tin sinh viên. Thời hạn: trong 7 ngày theo lịch đăng ký của khoa. Lệ phí: miễn phí.\nCâu hỏi: Thủ tục đăng ký học phần học kỳ chính làm tại đơn vị nào?\nTrả lời ngắn gọn dựa trên ngữ cảnh.",
 "chosen": "Theo quy định, đơn vị tiếp nhận là: cổng thông tin sinh viên.",
 "rejected": "Theo quy định, đơn vị tiếp nhận là: Phòng Đào tạo."
}


## Phần F – Đánh giá, ablation, phân tích lỗi
### F1. Ablation **A4**: quét trọng số hybrid α = 0,0; 0,1; …; 1,0 trên tập dev
"Bỏ / đổi 1 yếu tố": α = 0 là **bỏ dense** (chỉ còn BM25), α = 1 là **bỏ BM25** (chỉ còn dense), 0 < α < 1 là kết hợp cả hai. Bảng quét trên dev đã tính ở C2; test chỉ dùng để báo cáo.

In [20]:
print("Quét α trên DEV (MRR@10 cao nhất tại α =", alpha_best, "):")
print(sweep_dev[["recall@1", "recall@3", "mrr@10", "ndcg@10"]].round(3))

cot = ["recall@1", "recall@3", "mrr@10", "ndcg@10"]
bang_ablation = pd.DataFrame({
    "α=0 (bỏ dense, chỉ BM25) – clean": ret["bm25_clean"],  f"α={alpha_best} (hybrid) – clean": ret["hybrid_clean"],  "α=1 (bỏ BM25, chỉ dense) – clean": ret["dense_clean"],
    "α=0 (bỏ dense, chỉ BM25) – noisy": ret["bm25_noisy"],  f"α={alpha_best} (hybrid) – noisy": ret["hybrid_noisy"],  "α=1 (bỏ BM25, chỉ dense) – noisy": ret["dense_noisy"],
}).T[cot]
print("\nBáo cáo trên retr_test:")
print(bang_ablation.round(3))

Quét α trên DEV (MRR@10 cao nhất tại α = 0.7 ):
       recall@1  recall@3  mrr@10  ndcg@10
alpha                                     
0.0       0.629     0.733   0.699    0.741
0.1       0.629     0.724   0.705    0.747
0.2       0.621     0.759   0.711    0.758
0.3       0.629     0.767   0.721    0.769
0.4       0.638     0.767   0.729    0.781
0.5       0.629     0.784   0.732    0.788
0.6       0.629     0.810   0.737    0.792
0.7       0.629     0.802   0.738    0.793
0.8       0.612     0.784   0.718    0.773
0.9       0.560     0.750   0.673    0.730
1.0       0.491     0.681   0.608    0.669

Báo cáo trên retr_test:
                                  recall@1  recall@3  mrr@10  ndcg@10
α=0 (bỏ dense, chỉ BM25) – clean     0.727     0.841   0.804    0.844
α=0.7 (hybrid) – clean               0.758     0.909   0.838    0.874
α=1 (bỏ BM25, chỉ dense) – clean     0.576     0.758   0.674    0.730
α=0 (bỏ dense, chỉ BM25) – noisy     0.644     0.818   0.750    0.801
α=0.7 (hybrid) – n

**Giải thích F1.**
- Trên dev, MRR@10 tăng dần từ 0,699 (α = 0, chỉ BM25) lên cao nhất 0,738 tại **α = 0,7**, rồi giảm mạnh xuống 0,608 khi α = 1 (chỉ dense). Kết hợp tốt hơn từng thành phần riêng lẻ.
- Trên test sạch kết luận giữ nguyên: bỏ dense còn 0,804, bỏ BM25 còn 0,674, hybrid đạt 0,838. Bỏ BM25 mất nhiều hơn (−0,164) so với bỏ dense (−0,034), tức BM25 là thành phần quan trọng hơn trên bộ dữ liệu này.
- Lý do: BM25 khớp chính xác tên thủ tục / tên đơn vị, dense bắt được cách nói đời thường và từ tiếng Anh mà BM25 không khớp; hai tín hiệu sai ở những câu khác nhau nên bù cho nhau. Rõ nhất ở tập nhiễu: hybrid đạt 0,832 trong khi BM25 chỉ 0,750 và dense 0,649.

### F2. Confusion matrix cho `topic_pair` và phân loại nguyên nhân lỗi
Xét hai bộ phân loại đã làm (B1: TF-IDF + LR, C3: Embedding + LR) trên các câu test thuộc hai chủ đề `thi_diem` và `thu_vien_cntt`. Hàng = nhãn thật, cột = nhãn dự đoán.

In [21]:
pair = card["topic_pair"]
du_doan = {"TF-IDF+LR": pred_tfidf, "Embedding+LR": pred_emb}

ds_loi = []                                                     # gom mọi câu sai thuộc topic_pair
for mo_hinh, pred in du_doan.items():
    for tap, df in [("clean", te), ("noisy", te_noisy)]:
        y_that, y_doan = df.topic.values, pred[tap]
        thuoc_cap = np.isin(y_that, pair)                       # chỉ lấy câu có nhãn thật thuộc topic_pair
        print(f"\nConfusion matrix – {mo_hinh} – {tap}:")
        print(pd.crosstab(pd.Series(y_that[thuoc_cap], name="thật"), pd.Series(y_doan[thuoc_cap], name="dự đoán")))
        for i in np.where(thuoc_cap & (y_doan != y_that))[0]:
            ds_loi.append({"mô hình": mo_hinh, "tập": tap, "qid": df.qid.iloc[i], "câu hỏi": df.text.iloc[i],
                           "thật": y_that[i], "đoán": y_doan[i],
                           "đúng khi chưa nhiễu": bool(pred["clean"][i] == y_that[i])})
loi = pd.DataFrame(ds_loi)
print("\nSố câu sai thuộc topic_pair:")
print(loi.groupby(["mô hình", "tập"]).size())


Confusion matrix – TF-IDF+LR – clean:
dự đoán        hoc_phi  thi_diem  thu_vien_cntt  tot_nghiep
thật                                                       
thi_diem             0        48              0           0
thu_vien_cntt        1        19             24           4

Confusion matrix – TF-IDF+LR – noisy:
dự đoán        dang_ky_hoc_phan  hoc_phi  thi_diem  thu_vien_cntt  tot_nghiep
thật                                                                         
thi_diem                      0        0        48              0           0
thu_vien_cntt                 1        1        16             24           6

Confusion matrix – Embedding+LR – clean:
dự đoán        ky_tuc_xa  thi_diem  thu_vien_cntt
thật                                             
thi_diem               4        44              0
thu_vien_cntt         10         0             38

Confusion matrix – Embedding+LR – noisy:
dự đoán        ctsv  ky_tuc_xa  thi_diem  thu_vien_cntt  tot_nghiep
thật              

In [22]:
# Nhóm đọc từng câu sai và tự phân loại nguyên nhân. Khóa: (mô hình, tập, qid) -> (nguyên nhân, giải thích)
NGUYEN_NHAN = {
    ("TF-IDF+LR", "clean", "Q066-01"):    ("từ khóa gây nhầm", "'cấp lại' trong train chỉ xuất hiện ở thi_diem (cấp lại phiếu dự thi) nên kéo câu về thi_diem"),
    ("TF-IDF+LR", "clean", "Q066-17"):    ("từ khóa gây nhầm", "giống câu trên: 'cấp lại' + 'lệ phí' là các từ đã gặp ở thi_diem; 'mật khẩu' là từ lạ nên không giúp được"),
    ("TF-IDF+LR", "clean", "Q066-05"):    ("thiếu ngữ cảnh", "'quên mật khẩu', 'đăng nhập' chưa từng có trong train; phần còn lại ('làm ở đâu') không chỉ ra chủ đề"),
    ("Embedding+LR", "clean", "Q066-22"): ("thiếu ngữ cảnh", "câu nói đời thường, không nhắc 'cổng thông tin' hay 'CNTT'; train không có mục nào về mật khẩu"),
    ("Embedding+LR", "clean", "Q032-06"): ("mơ hồ", "'bị ốm … liên hệ phòng nào' có thể hiểu là chuyện sức khỏe / sinh hoạt, không nhắc 'hoãn thi'"),
    ("Embedding+LR", "noisy", "Q031-03"): ("nhiễu", "'hồ sơ' bị đổi thành 'document'; câu sạch dự đoán đúng, câu nhiễu thành tot_nghiep"),
    ("Embedding+LR", "noisy", "Q032-16"): ("nhiễu", "'nộp hồ sơ' bị đổi thành 'submit document'; câu sạch đúng, câu nhiễu thành ctsv"),
    ("Embedding+LR", "noisy", "Q066-09"): ("nhiễu", "'Thời hạn' bị đổi thành 'deadline'; câu sạch đúng, câu nhiễu thành thi_diem"),
}

khoa = list(zip(loi["mô hình"], loi["tập"], loi["qid"]))
loi["nguyên nhân"] = [NGUYEN_NHAN.get(k, ("", ""))[0] for k in khoa]
loi["giải thích"] = [NGUYEN_NHAN.get(k, ("", ""))[1] for k in khoa]

da_phan_loai = loi[loi["nguyên nhân"] != ""]
print(f"{len(da_phan_loai)} câu sai đã được phân loại nguyên nhân:")
print(da_phan_loai[["mô hình", "tập", "câu hỏi", "thật", "đoán", "nguyên nhân", "giải thích"]].to_string(index=False))

# Kiểm tra khách quan cho nhãn "nhiễu": câu đó phải đúng khi chưa nhiễu và sai sau khi nhiễu
assert da_phan_loai[da_phan_loai["nguyên nhân"] == "nhiễu"]["đúng khi chưa nhiễu"].all()

8 câu sai đã được phân loại nguyên nhân:
     mô hình   tập                                                                câu hỏi          thật       đoán      nguyên nhân                                                                                                giải thích
   TF-IDF+LR clean              Muốn cấp lại mật khẩu cổng thông tin thì thực hiện ở đâu? thu_vien_cntt   thi_diem từ khóa gây nhầm             'cấp lại' trong train chỉ xuất hiện ở thi_diem (cấp lại phiếu dự thi) nên kéo câu về thi_diem
   TF-IDF+LR clean                Em quên mật khẩu không đăng nhập được, thì làm ở đâu ạ? thu_vien_cntt   thi_diem   thiếu ngữ cảnh     'quên mật khẩu', 'đăng nhập' chưa từng có trong train; phần còn lại ('làm ở đâu') không chỉ ra chủ đề
   TF-IDF+LR clean                   Lệ phí cấp lại mật khẩu cổng thông tin là bao nhiêu? thu_vien_cntt   thi_diem từ khóa gây nhầm giống câu trên: 'cấp lại' + 'lệ phí' là các từ đã gặp ở thi_diem; 'mật khẩu' là từ lạ nên không giúp được
Embeddi

**Nhận xét F2.**
- **`thu_vien_cntt`**: gần như toàn bộ lỗi rơi vào **một mục** – *Cấp lại mật khẩu cổng thông tin* (Q066). Train không có mục nào nói về mật khẩu. TF-IDF + LR đoán nhầm sang `thi_diem` (đúng là nhầm giữa hai chủ đề của `topic_pair`) vì từ *"cấp lại"* trong train chỉ gặp ở *cấp lại phiếu dự thi*. Mục còn lại (*Gia hạn sách đã mượn*) được đoán đúng nhờ từ *"sách"*, *"mượn"* rất đặc trưng.
- **`thi_diem`**: TF-IDF đúng hết; Embedding sai ở các câu đời thường *"bị ốm không đi thi được"* (mục *Hoãn thi*) – câu không nhắc từ *"hoãn thi"* nên bị hiểu sang chủ đề sinh hoạt.
- **Nhiễu**: 4 câu Embedding đúng ở tập sạch nhưng sai ở tập nhiễu, đều là câu có từ bị thay (*hồ sơ → document, nộp → submit, thời hạn → deadline*).
- Bài học: lỗi chủ yếu do **mục test có từ vựng chưa từng gặp** (hệ quả của việc tách theo mục), không phải do hai chủ đề giống nhau về nội dung.

### F3. Tự viết 10 câu hỏi mới cho mỗi chủ đề trong `topic_pair` → `own_queries.csv`, đo Recall@3

In [23]:
OWN_QUERIES = [    # (câu hỏi, chủ đề, đoạn quy định đúng)
    # ---- thi_diem (P031–P040) ----
    ("Điểm cuối kỳ môn Giải tích lệch xa so với mình tự ước lượng, có cách nào xin xem xét lại bài làm không?", "thi_diem", "P031"),
    ("Hôm thi bị sốt cao phải vào bệnh viện, mình có xin dời sang lần thi khác được không?",                   "thi_diem", "P032"),
    ("Trượt môn Xác suất thống kê thì ghi danh thi lần hai trong khoảng thời gian nào?",                      "thi_diem", "P033"),
    ("Cần bản in toàn bộ điểm các học kỳ để nộp hồ sơ xin việc thì bao lâu sẽ nhận được?",                    "thi_diem", "P034"),
    ("Mình xin bảng điểm bằng tiếng Anh để nộp hồ sơ du học thì mỗi bản tính phí bao nhiêu?",                 "thi_diem", "P035"),
    ("Điểm kiểm tra giữa kỳ bị chấm thiếu ý, mình phản ánh với ai và trong bao nhiêu ngày?",                  "thi_diem", "P036"),
    ("Khi nào em đủ điều kiện ghi danh kỳ thi chuẩn ngoại ngữ để ra trường?",                                  "thi_diem", "P037"),
    ("Làm mất giấy báo dự thi sát ngày thi thì xin cấp lại ở đâu?",                                            "thi_diem", "P038"),
    ("Em đang thực tập xa trường, muốn thi ở địa điểm khác với lịch của khoa thì phải báo trước bao lâu?",     "thi_diem", "P039"),
    ("Xem thời gian các buổi thi cuối kỳ ở trang nào của trường?",                                             "thi_diem", "P040"),
    # ---- thu_vien_cntt (P061–P070) ----
    ("Sinh viên mới nhập học muốn có thẻ để vào đọc sách ở thư viện thì cần làm gì?",                          "thu_vien_cntt", "P061"),
    ("Mình được giữ cuốn giáo trình ở nhà tối đa mấy ngày?",                                                    "thu_vien_cntt", "P062"),
    ("Sách sắp đến hạn trả mà mình chưa đọc xong, xin mượn thêm hạn vào lúc nào thì được?",                    "thu_vien_cntt", "P063"),
    ("Lỡ làm thất lạc quyển sách mượn từ thư viện thì phải đền thế nào?",                                       "thu_vien_cntt", "P064"),
    ("Nhóm 5 người tụi mình muốn có phòng riêng để làm đồ án ở thư viện, phải giữ chỗ sớm ra sao?",            "thu_vien_cntt", "P065"),
    ("Quên mật khẩu đăng nhập hệ thống của sinh viên thì liên hệ bộ phận nào để đặt lại?",                     "thu_vien_cntt", "P066"),
    ("Muốn kết nối mạng không dây trong khuôn viên trường thì đăng ký ở đâu và có tốn phí không?",             "thu_vien_cntt", "P067"),
    ("Khi nào mình sẽ có địa chỉ thư điện tử của trường sau khi nhập học?",                                    "thu_vien_cntt", "P068"),
    ("Cần cài bộ phần mềm trường đã mua bản quyền lên laptop cá nhân thì nộp yêu cầu như thế nào?",            "thu_vien_cntt", "P069"),
    ("Ở nhà vẫn muốn đọc tạp chí khoa học của thư viện điện tử, có cách kết nối từ xa không?",                 "thu_vien_cntt", "P070"),
]
own = pd.DataFrame(OWN_QUERIES, columns=["text", "topic", "gold_passage_id"])

# Kiểm tra: đủ 10 câu mỗi chủ đề và không câu nào trùng với câu có sẵn trong dữ liệu
assert (own.topic.value_counts() == 10).all() and set(own.topic) == set(pair)
cau_co_san = {normalize(t) for t in pd.concat([tr.text, te.text, dev.text, rq.text, llm_eval.text, fewshot.text])}
assert not any(normalize(t) in cau_co_san for t in own.text), "có câu trùng với dữ liệu có sẵn"
own.to_csv("own_queries.csv", index=False, encoding="utf-8-sig")
print("Đã lưu own_queries.csv:", own.shape)

# Đo Recall@3 của hệ thống truy hồi (hybrid, α đã chốt trên dev)
H_own = hybrid_scores(dense_matrix(own.text), bm25_matrix(own.text), alpha_best)
own["hạng đoạn đúng"] = [rank_of_gold(H_own[i], pid2idx[g]) for i, g in enumerate(own.gold_passage_id)]
own_recall3 = float((own["hạng đoạn đúng"] <= 3).mean())
own_recall3_by_topic = {t: float((own[own.topic == t]["hạng đoạn đúng"] <= 3).mean()) for t in pair}
print(f"Recall@3 trên 20 câu tự viết = {own_recall3:.3f} | theo chủ đề: {own_recall3_by_topic}")
print("\nCác câu mà đoạn đúng không vào top-3:")
print(own[own["hạng đoạn đúng"] > 3][["text", "gold_passage_id", "hạng đoạn đúng"]].to_string(index=False))

Đã lưu own_queries.csv: (20, 3)
Recall@3 trên 20 câu tự viết = 0.900 | theo chủ đề: {'thi_diem': 0.8, 'thu_vien_cntt': 1.0}

Các câu mà đoạn đúng không vào top-3:
                                                                                              text gold_passage_id  hạng đoạn đúng
Em đang thực tập xa trường, muốn thi ở địa điểm khác với lịch của khoa thì phải báo trước bao lâu?            P039               5
                                        Xem thời gian các buổi thi cuối kỳ ở trang nào của trường?            P040               4


**Nhận xét F3.** Recall@3 = 0,90 (18/20): `thu_vien_cntt` 10/10, `thi_diem` 8/10. Hai câu trượt top-3 đều dùng cách nói khác hẳn tên thủ tục: *"thi ở địa điểm khác"* (đoạn đúng viết *"dự thi tại điểm thi khác"*) và *"thời gian các buổi thi … ở trang nào"* (đoạn đúng viết *"tra cứu lịch thi"*), trong khi nhiều đoạn khác của `thi_diem` cũng chứa từ *"thi"* nên cạnh tranh điểm.

### F4. Hạn chế, rủi ro và cải tiến
- **Hạn chế 1 – dữ liệu:** KB chỉ có 80 đoạn *giả lập* viết theo hai khuôn câu cố định; câu hỏi cũng sinh từ khuôn mẫu nên kết quả chưa chắc đúng với văn bản quy chế thật (dài, tự do).
- **Hạn chế 2 – đánh giá:** tập dev/test nhỏ (116 và 132 câu, 22 mục test) nên α và τ chọn trên dev dao động lớn; mới thử một kiểu nhiễu (`code_mix`).
- **Hạn chế 3 – mô hình:** RAG trích xuất bằng regex nên sẽ hỏng nếu đoạn quy định đổi cách viết; ngưỡng từ chối chỉ bắt được 3/10 câu ngoài phạm vi.
- **Rủi ro – ảo giác (hallucination):** khi truy hồi sai đoạn, hệ thống vẫn trả lời *tự tin* bằng đơn vị / lệ phí của một quy định khác (ở D2 có khoảng 12 % câu trả lời sai), và các câu ngoài phạm vi nhưng gần chủ đề không bị chặn. Sinh viên có thể làm theo thông tin sai.
- **Cải tiến sẽ làm tiếp:** fine-tune mô hình embedding bằng các cặp (câu hỏi, đoạn đúng) của train, kèm đoạn sai cùng chủ đề làm ví dụ khó, để tăng chất lượng dense; khi đó cả hybrid lẫn ngưỡng từ chối (dựa trên cosine) đều được cải thiện.

## Kết quả nộp: `results.json` và `report.md`

In [24]:
def save_results(results: dict, path="results.json"):
    results = {"mssv": card["mssv"], "check_code": card["check_code"], **results}
    json.dump(results, open(path, "w", encoding="utf-8"), ensure_ascii=False, indent=2, default=float)
    print("Đã lưu", path)

E_truong = ket_qua_E[MSSV]            # Phần E theo thẻ nhóm trưởng (thẻ chung của sản phẩm)
save_results({
    "group": [{"ho_ten": ho_ten, "mssv": mssv, "vai_tro": vai_tro} for ho_ten, mssv, vai_tro, _ in NHOM],
    "clf": {"tfidf_lr": res, "emb_lr": res_emb, "lora": lora_info},
    "retrieval": {"bm25_clean": ret["bm25_clean"], "bm25_noisy": ret["bm25_noisy"],
                  "dense_clean": ret["dense_clean"], "dense_noisy": ret["dense_noisy"],
                  "hybrid": {"clean": ret["hybrid_clean"], "noisy": ret["hybrid_noisy"]}, "alpha_best": alpha_best},
    "prompting": {"model": LLM_NAME, "revision": LLM_VERSION, "temperature": 0, "fewshot_k": K_SHOT,
                  "zero_shot_acc": zero_shot_acc, "k_shot_acc": k_shot_acc},
    "rag": {"top_k": RAG_K, "n_questions": len(rag), "answer_hit_rate": answer_hit_rate,
            "tau": tau, "abstain_rate_ood": abstain_rate_ood, "false_abstain_rate": false_abstain_rate},
    "toy": {"ranking": {m: E_truong[m] for m in ("P@5", "R@5", "RR", "nDCG@5")},
            "dpo": {"margin_h": E_truong["h"], "loss": E_truong["loss"]},
            "lora_params": E_truong["lora"]},
    "toy_members": {mssv: {k: v for k, v in kq.items() if k not in ("ho_ten", "backbone")} for mssv, kq in ket_qua_E.items()},
    "ablation": {"id": card["ablation"], "desc": card["ablation_desc"],
                 "table": {"dev_mrr@10_theo_alpha": {str(a): float(sweep_dev.loc[a, "mrr@10"]) for a in ALPHAS},
                           "test_clean_mrr@10": {"alpha=0 (BM25)": ret["bm25_clean"]["mrr@10"], f"alpha={alpha_best} (hybrid)": ret["hybrid_clean"]["mrr@10"], "alpha=1 (dense)": ret["dense_clean"]["mrr@10"]}}},
    "own_queries": {"n": len(own), "recall@3": own_recall3, "recall@3_by_topic": own_recall3_by_topic},
})

Đã lưu results.json


In [25]:
def bang_md(df, so_le=3):
    """DataFrame -> bảng Markdown."""
    dong = ["| | " + " | ".join(str(c) for c in df.columns) + " |", "|---" * (len(df.columns) + 1) + "|"]
    for ten, hang in df.iterrows():
        dong.append(f"| {ten} | " + " | ".join(f"{v:.{so_le}f}" for v in hang) + " |")
    return "\n".join(dong)

bang_clf = pd.DataFrame({"TF-IDF+LR": {"acc sạch": res["clean"]["acc"], "F1 sạch": res["clean"]["macro_f1"], "acc nhiễu": res["noisy"]["acc"], "F1 nhiễu": res["noisy"]["macro_f1"]},
                         "Embedding+LR": {"acc sạch": res_emb["clean"]["acc"], "F1 sạch": res_emb["clean"]["macro_f1"], "acc nhiễu": res_emb["noisy"]["acc"], "F1 nhiễu": res_emb["noisy"]["macro_f1"]}}).T
bang_ret = pd.DataFrame({"BM25 – sạch": ret["bm25_clean"], "BM25 – nhiễu": ret["bm25_noisy"],
                         "Dense – sạch": ret["dense_clean"], "Dense – nhiễu": ret["dense_noisy"],
                         f"Hybrid α={alpha_best} – sạch": ret["hybrid_clean"], f"Hybrid α={alpha_best} – nhiễu": ret["hybrid_noisy"]}).T
dong_lora = (f"LoRA `{card['clf_backbone']}` (C4): {lora_info['trainable_params']:,} tham số huấn luyện được, acc sạch {lora_info['clean']['acc']:.3f} / nhiễu {lora_info['noisy']['acc']:.3f}."
             if lora_info["trainable_params"] else "LoRA (C4, nâng cao): chưa chạy vì không có GPU.")
dong_E = "\n".join(f"| {kq['ho_ten']} ({mssv}) | {kq['lora']:,} | {kq['h']:.2f} | {kq['loss']:.4g} | {kq['P@5']:.2f} | {kq['R@5']:.2f} | {kq['RR']:.2f} | {kq['nDCG@5']:.4f} |"
                   for mssv, kq in ket_qua_E.items())
dong_nhom = "\n".join(f"| {ho_ten} | {mssv} | {vai_tro} | {phan} |" for ho_ten, mssv, vai_tro, phan in NHOM)
m = lambda he, tap: ret[f"{he}_{tap}"]["mrr@10"]          # viết tắt: MRR@10 của một hệ truy hồi

report = f"""# Báo cáo Lab 9 – SSA-VN (Modern NLP)
**MSSV nhóm trưởng (thẻ chung): {MSSV}** · check_code `{card['check_code']}` · seed {SEED} · noise `{card['noise']}` · topic_pair {pair}  
**Nhóm 3 sinh viên:** {", ".join(f"{ho_ten} ({mssv})" for ho_ten, mssv, _, _ in NHOM)}  
**Cấu hình thẻ:** embed `{card['embed_model']}`; backbone `{card['clf_backbone']}` (LoRA r={card['lora']['r']}, alpha={card['lora']['alpha']}); few-shot k={K_SHOT}; rag_top_k={RAG_K}; ablation {card['ablation']}.

## 1. Bảng kết quả
**Phân loại chủ đề (clf_test):**

{bang_md(bang_clf)}

{dong_lora}

**Truy hồi (retr_test):**

{bang_md(bang_ret)}

**LLM và RAG:** zero-shot = {zero_shot_acc:.3f}, {K_SHOT}-shot = {k_shot_acc:.3f} (`{LLM_NAME}`, temperature 0, 30 câu); answer_hit_rate = {answer_hit_rate:.3f} ({len(rag)} câu, top-{RAG_K}); ngưỡng τ = {tau:.3f}: abstain_rate_ood = {abstain_rate_ood:.3f}, false_abstain_rate = {false_abstain_rate:.3f}. Recall@3 trên 20 câu tự viết = {own_recall3:.3f}.

**Phần E (mỗi thành viên một thẻ, code khớp lời giải tay):**

| Thành viên | E1: tham số LoRA | E2: margin h | E2: loss | P@5 | R@5 | RR | nDCG@5 |
|---|---|---|---|---|---|---|---|
{dong_E}

## 2. Ba phát hiện chính
1. **Hybrid tốt hơn từng thành phần và bền với nhiễu.** MRR@10 trên retr_test sạch: BM25 {m('bm25', 'clean'):.3f}, dense {m('dense', 'clean'):.3f}, hybrid {m('hybrid', 'clean'):.3f}. Sang tập nhiễu `{card['noise']}`, BM25 giảm {m('bm25', 'clean') - m('bm25', 'noisy'):.3f}, dense giảm {m('dense', 'clean') - m('dense', 'noisy'):.3f}, hybrid chỉ giảm {m('hybrid', 'clean') - m('hybrid', 'noisy'):.3f}.
2. **Nhiễu code_mix làm hỏng mô hình khớp từ, ít ảnh hưởng embedding.** TF-IDF+LR giảm accuracy từ {res['clean']['acc']:.3f} xuống {res['noisy']['acc']:.3f} vì từ tiếng Anh là từ lạ; Embedding+LR chỉ giảm từ {res_emb['clean']['acc']:.3f} xuống {res_emb['noisy']['acc']:.3f}. Tách theo mục khiến bài toán phân loại khó: lỗi tập trung ở các mục test có từ vựng chưa gặp trong train.
3. **RAG trả lời đúng {answer_hit_rate:.0%} số câu nhưng ngưỡng từ chối còn yếu.** Ngưỡng τ chốt trên dev chỉ chặn được {abstain_rate_ood:.0%} câu ngoài phạm vi (các câu hoàn toàn lạc đề) và từ chối nhầm {false_abstain_rate:.1%} câu hợp lệ; câu ngoài phạm vi nhưng gần chủ đề có cosine cao như câu hợp lệ.

## 3. Ablation {card['ablation']} – {card['ablation_desc']}
MRR@10 trên dev theo α: {", ".join(f"{a}: {sweep_dev.loc[a, 'mrr@10']:.3f}" for a in ALPHAS)} → chọn **α = {alpha_best}**.
Trên retr_test sạch: bỏ dense (α=0) {m('bm25', 'clean'):.3f}; hybrid {m('hybrid', 'clean'):.3f}; bỏ BM25 (α=1) {m('dense', 'clean'):.3f}. Bỏ BM25 mất {m('hybrid', 'clean') - m('dense', 'clean'):.3f}, bỏ dense mất {m('hybrid', 'clean') - m('bm25', 'clean'):.3f}: BM25 quan trọng hơn vì câu hỏi hay nhắc đúng tên thủ tục, dense bổ sung cho các câu nói đời thường và từ tiếng Anh.

## 4. Hạn chế, rủi ro, cải tiến
Hạn chế: dữ liệu giả lập, KB 80 đoạn theo khuôn mẫu; dev/test nhỏ nên α, τ dao động; RAG trích xuất bằng regex phụ thuộc cách viết của đoạn. Rủi ro: ảo giác – khi truy hồi sai, hệ thống vẫn trả lời tự tin bằng thông tin của quy định khác. Cải tiến: fine-tune mô hình embedding bằng cặp (câu hỏi, đoạn đúng) của train để cải thiện dense, hybrid và ngưỡng từ chối.

## 5. Nhật ký sử dụng AI
- **Công cụ:** Claude (Anthropic).
- **AI hỗ trợ:** đọc đề; viết và rút gọn code các phần C–F trong notebook; soạn lời giải tham khảo E1–E3 để đối chiếu; gợi ý 20 câu hỏi `own_queries.csv`; soạn nháp báo cáo (số liệu sinh tự động từ notebook).
- **Giữ nguyên code của giảng viên:** `normalize`, `BM25`, các hàm metric truy hồi, baseline TF-IDF+LR, `save_results`.
- **Nhóm tự làm và kiểm chứng:** chạy lại toàn bộ notebook, làm tay E1–E3 theo thẻ riêng, đọc và phân loại nguyên nhân các câu sai; α và τ chỉ chọn trên dev.

## 6. Phân công công việc
| Họ tên | MSSV | Vai trò | Phần phụ trách |
|---|---|---|---|
{dong_nhom}

Mọi thành viên đều chạy lại được notebook và giải thích được tất cả các phần.
"""
open("report.md", "w", encoding="utf-8").write(report)
print("Đã lưu report.md:", len(report.split()), "từ")

Đã lưu report.md: 920 từ
